# 06 — Step 5: vibe features

**Purpose.** Give every movie a description of how it *feels* (PROJECT.md §7: tone, pacing, ending, setting, characters, and so on). Steps 6 (vibe clusters), 9 (natural-language search), 10 (ranking) and 11 (the Torch button) are all built on it. **Pass/fail check (roadmap):** LLM vibe tags agree with the Tag Genome.

**Why an LLM is needed.** MovieLens's Tag Genome already scores movies on 1,128 descriptive tags, but it covers only 12,877 of the 34,403 approved movies (37%), and none with fewer than 28 ratings. The rest need vibe tags read from their plots.

| # | Task | Purpose | Status |
|---|---|---|---|
| 1 | Long plots: map each movie to its English Wikipedia page, then fetch the Plot section | The LLM needs real plot text; TMDB overviews are too short (decision 2026-10-08) | coverage check: this notebook |
| 2 | TMDB keywords | Extra signal for the LLM | waits for the API key |
| 3 | Explore the Tag Genome | Choose the tags the LLM is checked against | not started |
| 4 | Pilot LLM tagging (Bedrock) on about 200 movies with Genome scores | Agreement and cost before the full run | not started |
| 5 | Validate LLM tags vs Genome | The Step 5 pass/fail check | not started |
| 6 | Tag all movies, plot embeddings | Vibe for every movie | not started |

**Scope:** tier 2 and tier 3 movies from `movie_tiers_v2.parquet` (39,594). Tier 1 is excluded, so it is skipped.

## Config and load

In [1]:
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))   # lets the notebook import src/
from src.viberec import wikidata as wd

PROCESSED_DIR = Path("..") / "data" / "processed"
RAW_DIR = Path("..") / "data" / "raw"
VERSION = "v1"


def write_metadata(path, **info):
    """Small JSON next to a saved output: what produced it, when, with which settings."""
    meta = {"created_at": datetime.now(timezone.utc).isoformat(),
            "produced_by": "notebooks/06_vibe_features.ipynb", **info}
    path.with_name(path.stem + ".metadata.json").write_text(json.dumps(meta, indent=2, default=str))


tiers = pd.read_parquet(PROCESSED_DIR / "movie_tiers_v2.parquet")
pool = tiers[tiers["tier"].isin([2, 3])].copy()
print(f"{len(pool):,} movies in tiers 2 and 3; {pool['tconst'].notna().sum():,} have an IMDb id")

39,594 movies in tiers 2 and 3; 39,594 have an IMDb id


## 1a. Which movies have an English Wikipedia page?

Wikipedia does not use IMDb ids, and matching by title is unreliable (many movies share a title). Wikidata, Wikipedia's ID database, links IMDb ids (property P345) to Wikipedia articles. So one query per batch of 300 IMDb ids returns each movie's English Wikipedia article, if it has one.

**Crash-safe.** Each batch is saved to `data/raw/wikidata/imdb_enwiki_v1/` as soon as it arrives (written to a temp name, then renamed). A rerun skips the saved batches and continues from the first missing one. A failed request is retried with growing waits; if it keeps failing the run stops, and rerunning resumes it. A manifest file stops batches for a different movie list from being mixed in.

In [2]:
WIKIDATA_DIR = RAW_DIR / "wikidata" / f"imdb_enwiki_{VERSION}"
imdb_ids = pool["tconst"].dropna().tolist()

start = time.time()
fetched, skipped = wd.fetch_in_batches(imdb_ids, WIKIDATA_DIR, wd.sparql_imdb_to_enwiki, batch_size=300, pause=1.0)
print(f"fetched {fetched} batches, {skipped} already on disk, {time.time() - start:.0f}s")
if fetched:
    write_metadata(WIKIDATA_DIR / "batches.json", source="query.wikidata.org SPARQL (P345 -> enwiki sitelink)",
                   ids=len(set(imdb_ids)), batch_size=300, last_run_batches_fetched=fetched,
                   last_run_seconds=round(time.time() - start, 1))
matches = wd.load_batches(WIKIDATA_DIR)
print(f"{len(matches):,} rows for {matches['imdb'].nunique():,} IMDb ids")

fetched 0 batches, 132 already on disk, 0s
37,731 rows for 37,715 IMDb ids


In [3]:
# one row per movie: the Wikidata item and English article (ids matching several items keep the first with an article)
per_id = (matches.sort_values("article", na_position="last").drop_duplicates("imdb")
          .rename(columns={"imdb": "tconst", "item": "wikidata_item", "article": "enwiki_url"}))
print(f"IMDb ids matching more than one Wikidata item: {(matches.groupby('imdb')['item'].nunique() > 1).sum()}")

coverage = pool[["movieId", "tconst", "title", "tier", "rating_count"]].merge(per_id, on="tconst", how="left")
coverage["has_item"] = coverage["wikidata_item"].notna()
coverage["has_enwiki"] = coverage["enwiki_url"].notna()
print(coverage.groupby("tier")[["has_item", "has_enwiki"]].mean().round(3))
print(coverage.groupby(pd.cut(coverage["rating_count"], [0, 8, 28, 61, 1000, float("inf")], right=False))["has_enwiki"]
      .agg(["size", "mean"]).round(3))

MAP_PATH = PROCESSED_DIR / f"imdb_to_enwiki_{VERSION}.parquet"
if not MAP_PATH.exists():
    coverage.to_parquet(MAP_PATH, index=False)
    write_metadata(MAP_PATH, source=str(WIKIDATA_DIR), rows=len(coverage),
                   has_enwiki=int(coverage["has_enwiki"].sum()),
                   note="tier 2 + 3 movies with their Wikidata item and English Wikipedia URL (NaN = none found)")
    print(f"saved {len(coverage):,} rows to {MAP_PATH}")

IMDb ids matching more than one Wikidata item: 16
      has_item  has_enwiki
tier                      
2        0.856       0.662
3        0.967       0.896
                 size   mean
rating_count                
[0.0, 8.0)      17789  0.736
[8.0, 28.0)      7028  0.931
[28.0, 61.0)     3120  0.976
[61.0, 1000.0)   7900  0.992
[1000.0, inf)    3757  0.999


**Result.** 132 batches in 6.6 minutes.

| Group | Has an English Wikipedia page |
|---|---|
| Tier 3 (approved) | 89.6% |
| Tier 2 (reduced weight) | 66.2% |
| Fewer than 8 ratings | 73.6% |
| 8 to 27 ratings | 93.1% |
| 28 or more ratings | 97.6% to 99.9% |

About 34,000 of the 39,594 movies have a page. The gaps are mostly obscure movies with very few ratings. 16 IMDb ids match more than one Wikidata item; the one with an English article is kept.

**Caveat.** Having a page is not the same as having a plot: short stub pages often have no Plot section. The next check fetches a random sample of pages to measure how many have a usable plot, before the full fetch.

## 1b. How many Wikipedia pages actually have a plot? (sample)

A page is not a plot: short stub pages often have no Plot section. Before fetching all about 34,000 pages (a few hours), fetch a random sample and measure:
- the share of pages with a plot section (headed Plot, Plot summary, Synopsis, Premise, Story or Summary);
- how long the plots are.

The sample is 150 tier-3 and 150 tier-2 movies with a page. Tier 2 is sampled as heavily as tier 3 so it has enough movies to measure, even though it is the smaller group.

**Crash-safe, and nothing is fetched twice.** Each article is saved as `data/raw/wikipedia/extracts_v1/<movieId>.json` the moment it arrives. The full fetch later uses the same folder, so these 300 are not fetched again. The whole article text is cached, not just the plot, so the plot extraction can be changed without refetching.

In [4]:
from src.viberec import wikipedia as wp

EXTRACTS_DIR = RAW_DIR / "wikipedia" / f"extracts_{VERSION}"
SAMPLE_PER_TIER = 150
SAMPLE_SEED = 42

with_page = coverage[coverage["has_enwiki"]]
sample = pd.concat([with_page[with_page["tier"] == t].sample(SAMPLE_PER_TIER, random_state=SAMPLE_SEED) for t in (3, 2)])

start = time.time()
fetched, skipped = wp.fetch_each(dict(zip(sample["movieId"], sample["enwiki_url"].map(wp.title_from_url))),
                                 EXTRACTS_DIR, wp.fetch_extract, pause=0.5)
print(f"fetched {fetched}, already on disk {skipped}, {time.time() - start:.0f}s")

texts = {r["key"]: r["text"] for r in wp.load_extracts(EXTRACTS_DIR)}
sample["article_words"] = sample["movieId"].map(lambda k: len((texts.get(k) or "").split()))
sample["plot"] = sample["movieId"].map(lambda k: wp.plot_section(texts.get(k)))
sample["plot_words"] = sample["plot"].map(lambda t: len(t.split()) if isinstance(t, str) else 0)   # no plot -> 0
sample["has_plot"] = sample["plot_words"] > 0

print("\nshare with a plot section, and plot length in words (movies with a plot)")
print(sample.groupby("tier").agg(movies=("movieId", "size"), has_plot=("has_plot", "mean"))
      .join(sample[sample["has_plot"]].groupby("tier")["plot_words"].describe(percentiles=[0.1, 0.25, 0.5, 0.75])
            [["10%", "25%", "50%", "75%"]]).round(2))
print("\nshare with a plot, by number of ratings")
print(sample.groupby(pd.cut(sample["rating_count"], [0, 8, 28, 61, float("inf")], right=False))["has_plot"]
      .agg(["size", "mean"]).round(2))
print("\narticle length (words) for pages without a plot section:",
      sample.loc[~sample["has_plot"], "article_words"].describe(percentiles=[0.25, 0.5, 0.75])[["25%", "50%", "75%"]].to_dict())

fetched 0, already on disk 300, 0s

share with a plot section, and plot length in words (movies with a plot)
      movies  has_plot   10%    25%    50%     75%
tier                                              
2        150      0.80  31.9   60.0  139.0  361.50
3        150      0.89  71.1  180.5  459.0  651.75

share with a plot, by number of ratings
              size  mean
rating_count            
[0.0, 8.0)     196  0.81
[8.0, 28.0)     40  0.82
[28.0, 61.0)    17  0.94
[61.0, inf)     47  1.00

article length (words) for pages without a plot section: {'25%': 90.5, '50%': 108.5, '75%': 184.25}


**Result (sample).**

| | Tier 3 | Tier 2 |
|---|---|---|
| Pages with a plot section | 89% | 80% |
| Median plot length | 459 words | 139 words |
| Shortest quarter | under 180 words | under 60 words |

With the page coverage above, about 80% of tier-3 movies and about 53% of tier-2 movies get a Wikipedia plot. Pages without a plot section are stubs of about 100 words. Each figure is ±5 points or so (150 movies per tier).

**Problem.** Wikipedia rate-limited the one-page-per-request method (HTTP 429, "wait 14 seconds"). The 300 pages took 29 minutes, so all ~34,000 would take more than two days.

## 1c. Batched fetch: 50 pages per request

Wikipedia's API returns the raw page source ("wikitext") for up to 50 pages in one request, which needs about 680 requests instead of 34,000. The plot section is then cleaned with `mwparserfromhell`, the standard wikitext parser used by Wikipedia's own maintenance bots. It removes links, citations and templates, but keeps the text of small inline templates such as `{{nowrap|...}}`.

The fetch is crash-safe in the same way as the Wikidata step: each batch is saved to `data/raw/wikipedia/wikitext_v1/` as soon as it arrives, and a rerun skips saved batches. Wikipedia's "wait N seconds" replies are obeyed.

**Trial first.** `TRIAL_BATCHES = 20` (1,000 movies) measured the speed and the rate-limit hits; it is now `None` for the full fetch, and the 20 trial batches are not fetched again.

**Two sections are kept per movie:**
- **Plot:** what happens.
- **Reception:** how critics described the movie, for example "a bittersweet slice of observational comedy". This is the sentiment signal for the vibe and reviews vectors (PROJECT.md §8).

Both are saved to `data/processed/wiki_text_v1.parquet` once every batch is on disk.

In [5]:
import requests
from src.viberec import wikidata as wd

WIKITEXT_DIR = RAW_DIR / "wikipedia" / f"wikitext_{VERSION}"
TRIAL_BATCHES = None             # 20 for the trial run; None = fetch everything that is not on disk yet
titles = dict(zip(with_page["movieId"].astype(str), with_page["enwiki_url"].map(wp.title_from_url)))
calls = {"requests": 0, "rate_limited": 0}
session = requests.Session()


def fetch_batch(keys):
    calls["requests"] += 1
    try:
        rows = wp.fetch_wikitext_batch([titles[k] for k in keys], session)
    except wd.RetryableHTTPError:
        calls["rate_limited"] += 1
        raise
    return [{"movieId": k, **row} for k, row in zip(keys, rows)]


start = time.time()
fetched, skipped = wd.fetch_in_batches(list(titles), WIKITEXT_DIR, fetch_batch, batch_size=50, pause=1.0,
                                       retries=8, max_batches=TRIAL_BATCHES)
elapsed = time.time() - start
total_batches = -(-len(titles) // 50)
print(f"fetched {fetched} batches ({calls['requests']} requests, {calls['rate_limited']} rate-limited), "
      f"{skipped} already on disk, {elapsed:.0f}s")
if fetched:
    print(f"at this speed the remaining {total_batches - skipped - fetched} batches take about "
          f"{(total_batches - skipped - fetched) * elapsed / fetched / 60:.0f} minutes")
    write_metadata(WIKITEXT_DIR / "batches.json", source="en.wikipedia.org API, prop=revisions (wikitext), 50 titles per request",
                   movies=len(titles), batch_size=50, last_run_batches=fetched, last_run_seconds=round(elapsed, 1),
                   last_run_rate_limited=calls["rate_limited"])

wikitext = pd.DataFrame([row for path in sorted(WIKITEXT_DIR.glob("batch_*.json"))
                         for row in json.loads(path.read_text())["rows"]])
count_words = lambda t: len(t.split()) if isinstance(t, str) else 0
wikitext["plot"] = wikitext["wikitext"].map(wp.plot_from_wikitext)
wikitext["reception"] = wikitext["wikitext"].map(wp.reception_from_wikitext)    # critics' words: the sentiment signal
wikitext["plot_words"] = wikitext["plot"].map(count_words)
wikitext["reception_words"] = wikitext["reception"].map(count_words)
print(f"\n{len(wikitext):,} movies on disk; page found {wikitext['wikitext'].notna().mean():.1%}")
for part in ("plot", "reception"):
    has = wikitext[f"{part}_words"] > 0
    print(f"{part}: present {has.mean():.1%}, median {wikitext.loc[has, f'{part}_words'].median():.0f} words")

# save the cleaned text once every batch is on disk
WIKI_TEXT_PATH = PROCESSED_DIR / f"wiki_text_{VERSION}.parquet"
complete = wikitext["movieId"].nunique() == len(titles)
if complete and not WIKI_TEXT_PATH.exists():
    out = (wikitext.assign(movieId=wikitext["movieId"].astype(int))
           [["movieId", "title", "plot", "plot_words", "reception", "reception_words"]]
           .rename(columns={"title": "wiki_title"}))
    out.to_parquet(WIKI_TEXT_PATH, index=False)
    write_metadata(WIKI_TEXT_PATH, source=str(WIKITEXT_DIR), rows=len(out),
                   with_plot=int((out["plot_words"] > 0).sum()), with_reception=int((out["reception_words"] > 0).sum()),
                   note="cleaned Plot and Reception sections (mwparserfromhell), tier 2 + 3 movies with an English Wikipedia page")
    print(f"saved {len(out):,} rows to {WIKI_TEXT_PATH}")
elif not complete:
    print(f"{len(titles) - wikitext['movieId'].nunique():,} movies still to fetch; rerun this cell to continue")

fetched 0 batches (0 requests, 0 rate-limited), 686 already on disk, 0s

34,260 movies on disk; page found 100.0%
plot: present 88.5%, median 440 words
reception: present 71.3%, median 187 words


**Result (trial, 20 batches).** 1,000 movies in 122 seconds with no rate-limit hits, against 29 minutes for 300 movies before.
- Every page was found, 87.2% have a plot section, and the median plot is 450 words, in line with the sample.
- A spot check shows clean prose, and no plot has leftover markup (`{{`, `[[`, `<ref`).
- The remaining 666 batches should take about 68 minutes.

**Result (full fetch).** All 686 batches in 70 minutes. Wikipedia asked us to wait 37 times; the fetcher waited and retried each time, and nothing was lost. Saved: `data/processed/wiki_text_v1.parquet` (34,260 movies with a page).

| | Has a plot | Has Reception |
|---|---|---|
| Tier 3 (approved) | 80.2% | 66.1% |
| Tier 2 | 52.4% | 32.1% |
| Fewer than 8 ratings | 59.3% | 37.5% |
| 8–27 ratings | 81.8% | 61.5% |
| 28–60 ratings | 89.4% | 78.1% |
| 61+ ratings | 96.4% | 94.3% |

Median length: plot 440 words, Reception 187 words. This matches the sample estimate (about 80% of tier 3, 53% of tier 2).
- 12,356 approved movies have **both** a plot and Genome scores. That is the pool for the LLM pilot and the Genome check.
- 6,278 approved movies have **neither**. They will rely on TMDB text.

## 3. Tag Genome as the benchmark (vibe taxonomy)

**Purpose.** The Step 5 check compares LLM vibe tags with the Genome, so we need to know which Genome tags stand for which vibe. `src/viberec/genome.py` holds the taxonomy: 43 attributes across the §7 dimensions, each with its Genome tags.

**Check on the grouping.** It is a hand-made judgment, so it is tested with data. Tags grouped as synonyms should correlate across movies, measured as the mean Spearman correlation within each group, against random tag pairs as a baseline. Groups of alternatives (one city, one job) are scored as their highest tag, so they are not expected to correlate. `easy` and `comfort` failed the check, so they are validated with the hand-labeled set in `data/labels/vibe_labels_v1.xlsx` instead.

In [6]:
import numpy as np
from src.viberec import genome as gn

GENOME_PATH = PROCESSED_DIR / f"genome_matrix_{VERSION}.parquet"
ML_DIR = RAW_DIR / "movielens" / "ml-25m"
if GENOME_PATH.exists():
    print(f"{GENOME_PATH} already exists, loading instead of recomputing.")
    genome = pd.read_parquet(GENOME_PATH)
else:
    start = time.time()
    genome = gn.genome_matrix(ML_DIR / "genome-scores.csv", ML_DIR / "genome-tags.csv")
    genome.columns = genome.columns.astype(str)
    genome.to_parquet(GENOME_PATH)
    write_metadata(GENOME_PATH, source="ml-25m genome-scores.csv + genome-tags.csv", shape=list(genome.shape),
                   runtime_seconds=round(time.time() - start, 1))
print(f"Genome: {genome.shape[0]:,} movies x {genome.shape[1]:,} tags")

agreement, random_r = gn.within_vs_between(genome)
print(f"random tag pairs: median r {np.median(random_r):.2f}, 95th percentile {np.percentile(random_r, 95):.2f}\n")
print(agreement.round(2).to_string(index=False))

MAP_OUT = Path("..") / "artifacts" / f"genome_vibe_map_{VERSION}.csv"
if not MAP_OUT.exists():
    gn.map_table().merge(agreement, on="attribute").to_csv(MAP_OUT, index=False)
    write_metadata(MAP_OUT, random_pair_r_median=float(np.median(random_r)), random_pair_r_95th=float(np.percentile(random_r, 95)))

# sanity check: the top movies per attribute should look right
attr_scores = gn.attribute_scores(genome)
ml_titles = pd.read_csv(ML_DIR / "movies.csv").set_index("movieId")["title"]
print()
for a in ["dark_bleak", "heartwarming", "demanding", "underdog_rise", "twist", "workplace", "romantic", "easy", "comfort"]:
    print(f"{a:15s}", " | ".join(ml_titles.reindex(attr_scores[a].nlargest(4).index).str.slice(0, 30)))

../data/processed/genome_matrix_v1.parquet already exists, loading instead of recomputing.
Genome: 13,816 movies x 1,128 tags
random tag pairs: median r 0.12, 95th percentile 0.46

         attribute  tags  mean_within_r  min_pair_r      scoring  genome_checkable
        dark_bleak     6           0.69        0.57         mean              True
      heartwarming     6           0.65        0.40         mean              True
             tense     4           0.69        0.60         mean              True
       melancholic     6           0.65        0.50         mean              True
         whimsical     6           0.48        0.25         mean              True
             scary     2           0.63        0.63         mean              True
             funny     4           0.73        0.61         mean              True
       atmospheric     3           0.66        0.59         mean              True
              slow     4           0.64        0.55         mean        

**Result.** Random tag pairs correlate at 0.12 on average (95th percentile 0.46).
- **Synonym groups that pass:** 0.48 to 0.87, for example dark_bleak 0.69, romantic 0.74, dark humor 0.87. Their top movies are what you would expect: Se7en and Taxi Driver for dark; Amélie and Up for heartwarming; Inception and Primer for demanding; Rocky for underdog; The Usual Suspects for twist.
- **`easy` (0.26):** its top movies are flops, so it measures low quality rather than an easy watch.
- **`comfort` (0.38):** its tags do not agree with each other.
- **Outcome:** 41 of 43 attributes can be checked against the Genome; `easy` and `comfort` use the hand labels.

## 4. LLM pilot: 200 movies, checked against the Genome

**Purpose.** Before tagging ~34,000 movies, measure on 200 whether the LLM's vibe scores agree with the Genome, and what it costs. This is the Step 5 pass/fail check.

**Design**
- **Movies:** 200 approved movies that have both a Wikipedia plot and Genome scores, the same number from each decade band (<1970, 1970s, 1980s, 1990s, 2000s, 2010s). A plain random draw would be mostly recent movies.
- **Two variants:**
  - **A:** plot + reception.
  - **B:** A + TMDB keywords.

  Keywords stay only if B agrees with the Genome better than A.
- **Model:** Claude Haiku 4.5 on Bedrock (us-east-2, US cross-region profile), temperature 0. Bedrock's tool feature forces a fixed JSON answer (`src/viberec/vibe_llm.py`).
- **Answer:**
  - 41 attributes scored 0–3;
  - easy watch and comfort scored 1–5 (same scale as the hand-label sheet);
  - ending type, emotional arc, protagonist life stage and job;
  - a 100–150-word vibe paragraph.
- **Pass/fail:** per attribute, Spearman correlation between the LLM's score and the Genome's (`src/viberec/vibe_eval.py`). The LLM must beat a **genre-only baseline**, which guesses each attribute from the movie's genres. If it cannot, reading the plot added nothing for that attribute.

**Crash-safe.** Every answer is saved to `data/raw/bedrock/pilot_v1/<variant>/<movieId>.json` as soon as it arrives. A rerun skips saved answers, so nothing is paid for twice.

**Hand labels.** Adil's 99 labeled movies (`data/labels/vibe_labels_v2.xlsx`, sheet "Movie Ratings", rechecked by him) are tagged in the same run. They are the answer key for the three vibes the Genome cannot check: Light↔Heavy, Ending and Pacing. The sheet's scales are converted to the LLM's (light/heavy flipped so 5 = light; 7 endings mapped to happy / bittersweet / sad / open). The answers were drafted with an AI research tool and then checked by Adil, so very high agreement may partly be AI-to-AI agreement.

**Known bias.** The model may already know famous movies, which could inflate agreement for them. Results are reported by popularity too.

In [7]:
PILOT_N_PER_DECADE = 34          # 6 decade bands x 34 = 204, trimmed to 200
PILOT_PATH = Path("..") / "artifacts" / f"pilot_movies_{VERSION}.csv"
wiki_text = pd.read_parquet(PROCESSED_DIR / f"wiki_text_{VERSION}.parquet")

if PILOT_PATH.exists():
    print(f"{PILOT_PATH} already exists, loading instead of resampling.")
    pilot = pd.read_csv(PILOT_PATH)
else:
    candidates = (tiers[(tiers["tier"] == 3) & tiers["movieId"].isin(genome.index)]
                  .merge(wiki_text[wiki_text["plot_words"] > 0], on="movieId"))
    candidates["decade"] = pd.cut(candidates["year"], [0, 1970, 1980, 1990, 2000, 2010, 2030], right=False,
                                  labels=["<1970", "1970s", "1980s", "1990s", "2000s", "2010s"]).astype(str)
    pilot = (candidates.groupby("decade").sample(PILOT_N_PER_DECADE, random_state=42)
             .sample(frac=1, random_state=42).head(200))
    pilot = pilot[["movieId", "tmdbId", "title", "year", "genres", "rating_count", "decade"]]
    pilot.to_csv(PILOT_PATH, index=False)
    write_metadata(PILOT_PATH, pool=len(candidates), per_decade=PILOT_N_PER_DECADE, seed=42,
                   note="approved movies with a Wikipedia plot and Genome scores, equal per decade band")
print(pilot["decade"].value_counts().sort_index().to_dict())
print("rating count: median", int(pilot["rating_count"].median()), "| min", int(pilot["rating_count"].min()))

../artifacts/pilot_movies_v1.csv already exists, loading instead of resampling.
{'1970s': 33, '1980s': 34, '1990s': 33, '2000s': 33, '2010s': 34, '<1970': 33}
rating count: median 353 | min 45


In [8]:
from src.viberec import labels as lb

LABELS_PATH = PROCESSED_DIR / "vibe_labels_v2.parquet"
if LABELS_PATH.exists():
    print(f"{LABELS_PATH} already exists, loading.")
    labeled = pd.read_parquet(LABELS_PATH)
else:
    labeled = lb.match(lb.read_labels_v2(Path("..") / "data" / "labels" / "vibe_labels_v2.xlsx",
                                         title_fixes={"THE HOLMES?": "THE HELP"}),     # row 30, as noted in the sheet
                       pd.read_csv(ML_DIR / "movies.csv"))
    assert labeled["movieId"].notna().all() and not labeled["movieId"].duplicated().any()
    labeled["movieId"] = labeled["movieId"].astype(int)
    labeled.to_parquet(LABELS_PATH, index=False)
    write_metadata(LABELS_PATH, source="data/labels/vibe_labels_v2.xlsx (sheet Movie Ratings)", rows=len(labeled),
                   conversions={"light_heavy": "6 - sheet value (sheet: 1 = light; here: 5 = light)",
                                "ending": lb.ENDING_TO_FOUR}, title_fixes={"THE HOLMES?": "THE HELP (2011)"})
print(f"{len(labeled)} labeled movies; endings {labeled['ending'].value_counts().to_dict()}")

# movies the LLM tags in the pilot: the 200 Genome-check movies plus the labeled movies
run_set = pd.concat([pilot[["movieId", "tmdbId", "title", "year", "genres"]],
                     tiers.loc[tiers["movieId"].isin(labeled["movieId"]), ["movieId", "tmdbId", "title", "year", "genres"]]]
                    ).drop_duplicates("movieId").reset_index(drop=True)
print(f"{len(run_set)} movies in the pilot run ({len(set(labeled['movieId']) & set(pilot['movieId']))} in both sets)")

../data/processed/vibe_labels_v2.parquet already exists, loading.
99 labeled movies; endings {'happy': 57, 'sad': 19, 'open': 12, 'bittersweet': 11}
299 movies in the pilot run (0 in both sets)


In [9]:
from src.viberec import tmdb

TMDB_DIR = RAW_DIR / "tmdb" / f"movie_{VERSION}"
ids = run_set.dropna(subset=["tmdbId"])
fetched, skipped = wp.fetch_each(dict(zip(ids["tmdbId"].astype(int), ids["tmdbId"].astype(int))), TMDB_DIR,
                                 tmdb.fetch_movie, pause=0.1)
print(f"TMDB: fetched {fetched}, already on disk {skipped}")
tmdb_rows = {int(p.stem): json.loads(p.read_text()) for p in TMDB_DIR.glob("*.json")}
run_set["keywords"] = run_set["tmdbId"].map(lambda t: tmdb_rows.get(int(t), {}).get("keywords") if pd.notna(t) else None)
print(f"run movies with TMDB keywords: {run_set['keywords'].map(bool).sum()} of {len(run_set)}; "
      f"median keywords per movie: {run_set['keywords'].map(lambda k: len(k) if k else 0).median():.0f}")

TMDB: fetched 0, already on disk 299
run movies with TMDB keywords: 294 of 299; median keywords per movie: 13


### Run the pilot (needs the Bedrock quota)

`RUN_PILOT = False` keeps this cell from calling Bedrock until the quota increase is approved. With it on, each movie is one request per variant: about 300 movies (200 Genome-check + 99 labeled) × 2 variants, so about 600 requests. Each answer is saved as it arrives, and a rerun skips saved answers. Token usage is logged so the cost of the full run can be estimated.

In [10]:
from src.viberec import vibe_llm as vl

RUN_PILOT = False                       # True once the Bedrock quota is approved
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
PILOT_DIR = RAW_DIR / "bedrock" / f"pilot_{VERSION}"
VARIANTS = {"A_plot_reception": False, "B_plus_keywords": True}   # variant -> include TMDB keywords

texts = run_set.merge(wiki_text[["movieId", "plot", "reception"]], on="movieId")
if RUN_PILOT:
    import boto3
    client = boto3.Session(profile_name="viberec", region_name="us-east-2").client("bedrock-runtime")
    for variant, with_keywords in VARIANTS.items():
        prompts = {int(r.movieId): vl.build_prompt(r.title, r.year, r.genres, r.plot,
                                                   r.reception if isinstance(r.reception, str) else None,
                                                   r.keywords if with_keywords else None)
                   for r in texts.itertuples()}

        def ask(prompt):
            answer, usage = vl.call(client, MODEL_ID, prompt)
            return {"answer": answer, "usage": usage, "problems": vl.check_answer(answer), "model": MODEL_ID}

        start = time.time()
        fetched, skipped = wp.fetch_each(prompts, PILOT_DIR / variant, ask, pause=0.2)
        print(f"{variant}: {fetched} new answers, {skipped} already on disk, {time.time() - start:.0f}s")

saved = {v: [json.loads(p.read_text()) for p in (PILOT_DIR / v).glob("*.json")] for v in VARIANTS}
print({v: len(rows) for v, rows in saved.items()}, "answers on disk")

{'A_plot_reception': 0, 'B_plus_keywords': 0} answers on disk


### Evaluate: agreement with the Genome vs the genre-only baseline

In [11]:
from src.viberec import vibe_eval as ve

if all(len(rows) == len(texts) for rows in saved.values()):
    truth = attr_scores.loc[pilot["movieId"], list(vl.ATTRIBUTES)]
    genres_by_id = tiers.set_index("movieId")["genres"].combine_first(pd.read_csv(ML_DIR / "movies.csv").set_index("movieId")["genres"])
    baseline = ve.genre_baseline(attr_scores[list(vl.ATTRIBUTES)], genres_by_id, pilot["movieId"].tolist())
    results = {}
    for variant, rows in saved.items():
        llm = pd.DataFrame({r["key"]: r["answer"]["attributes"] for r in rows}).T.astype(float)
        llm.index = llm.index.astype(int)
        results[variant] = ve.agreement(llm[list(vl.ATTRIBUTES)], truth, baseline=baseline)
        r = results[variant]
        tokens = sum(x["usage"]["inputTokens"] + x["usage"]["outputTokens"] for x in rows)
        print(f"\n{variant}: median LLM rho {r['llm_rho'].median():.2f} vs baseline {r['baseline_rho'].median():.2f}; "
              f"beats baseline on {r['beats_baseline'].sum()} of {len(r)} attributes; "
              f"answers with problems: {sum(bool(x['problems']) for x in rows)}; tokens: {tokens:,}")
    both = results["A_plot_reception"].merge(results["B_plus_keywords"], on="attribute", suffixes=("_A", "_B"))
    print(both[["attribute", "llm_rho_A", "llm_rho_B", "baseline_rho_A", "beats_baseline_A", "beats_baseline_B"]].round(2).to_string(index=False))

    # hand labels: Light/Heavy, Ending, Pacing
    human = labeled.set_index("movieId")[["light_heavy", "pacing", "ending"]]
    for variant, rows in saved.items():
        llm_labels = pd.DataFrame({r["key"]: {"light_heavy": r["answer"]["light_heavy"], "pacing": r["answer"]["pacing"],
                                              "ending": r["answer"]["ending_type"]} for r in rows}).T
        llm_labels.index = llm_labels.index.astype(int)
        print(f"\n{variant} vs hand labels")
        print(ve.label_agreement(llm_labels, human, genres_by_id).round(2).to_string(index=False))
else:
    print("pilot answers not complete yet; run the cell above with RUN_PILOT = True once Bedrock is approved")

pilot answers not complete yet; run the cell above with RUN_PILOT = True once Bedrock is approved


## 5. Free alternative: learn vibes from text with our own model

**Why this section exists.** Section 4 (the Claude pilot) is built but **not run**: Adil chose not to pay for LLM tagging (2026-10-08). The free route is to train our own model.

**The idea.** 12,356 movies have both a Wikipedia text and Genome vibe scores. That is a ready-made training set: text in, vibe scores out. A model that learns the link from these movies can score the ~21,500 approved movies that have text but no Genome scores.

**Setup**
- **Input (X):** the plot, plus the critics' Reception section when there is one.
- **Target (Y):** the 41 Genome-checkable vibe attributes (0–1), the same ones validated in section 3.
- **Split:** 80% training, 20% test (seed 42). All tuning uses only the training movies. The test movies are scored once, at the end, so the result is honest.

**Contenders.** Simplest first; each must beat the one before (PROJECT.md §1):

| # | Model | Knows |
|---|---|---|
| 0 | Genre-only guess (baseline) | only the movie's genres |
| 1 | TF-IDF + ridge, plot only | the words of the plot |
| 2 | TF-IDF + ridge, plot + reception | plus how critics described it |
| 3 | TF-IDF + ridge, plot + reception **without review-score sentences** | the same, minus lines like "Metacritic … indicating universal acclaim" |

Model 3 was added after inspecting model 2: its strongest words for *dark*, *slow* and *melancholic* included "universal acclaim" and "unfavorable reviews". Those come from the review-score sentences in nearly every Reception section, so they measure **quality**, not vibe. The tiers already use quality, so vibe scores must not carry it too. Model 3 removes those sentences, and a separate check below measures how much each model's scores follow the movie's rating.

**How the model works**
- **TF-IDF** turns each text into one number per word (and two-word phrase). A word scores high when it is frequent in this text and rare across all texts, so "haunted" counts and "the" does not.
- **Ridge regression** learns one weight per word per vibe ("haunted" pushes *scary* up). It has a penalty, **alpha**, that keeps weights small unless the data supports them; without it, 50,000+ words and 10,000 movies would let the model memorise noise.
- The TF-IDF options and alpha are chosen by **3-fold cross-validation** on the training movies, scored by mean Spearman across the 41 vibes. None of them is set by hand.

**Measure:** per vibe, Spearman correlation between predicted and true Genome scores on the test movies. Differences between models get a paired bootstrap interval (resampling test movies).

In [12]:
from src.viberec import text_model as tm
from src.viberec import vibe_eval as ve

ATTRS = [a for a in attr_scores.columns if a not in gn.NOT_GENOME_CHECKABLE]       # the 41 checkable vibes
wt = wiki_text.set_index("movieId")
model_ids = wt.index[wt["plot_words"] > 0].intersection(attr_scores.index)          # have text AND Genome scores
train_ids, test_ids = tm.split_ids(model_ids, test_share=0.2, seed=42)
Y = attr_scores.loc[model_ids, ATTRS]
print(f"{len(model_ids):,} movies with a plot and Genome scores -> {len(train_ids):,} train, {len(test_ids):,} test")
print(f"{len(ATTRS)} target vibes; share of these movies with a Reception section: "
      f"{(wt.loc[model_ids, 'reception_words'] > 0).mean():.1%}")

# note: wt["plot"], not wt.plot (on a pandas table .plot means "draw a chart")
VARIANTS_TXT = {"plot": (False, False),                       # name -> (include Reception, strip review-score sentences)
                "plot_reception": (True, False),
                "plot_reception_noscores": (True, True)}
texts = {name: pd.Series({i: tm.movie_text(wt.at[i, "plot"], wt.at[i, "reception"], use_reception=rec, strip_scores=strip)
                          for i in model_ids})
         for name, (rec, strip) in VARIANTS_TXT.items()}

12,356 movies with a plot and Genome scores -> 9,845 train, 2,511 test
41 target vibes; share of these movies with a Reception section: 94.2%


### Train the two text models (settings chosen by cross-validation)

The search tries every combination below, each scored by 3-fold cross-validation on the training movies:
- single words only, or also two-word phrases (such as "serial killer" and "happy ending");
- a word must appear in at least 2 or at least 5 texts (rarer words are mostly names and typos);
- ridge strength alpha 0.3, 1, 3 or 10.

That is 16 combinations × 3 folds = 48 fits per model, a few minutes. If the best alpha lands on the edge of its range, the range is too narrow, and the cell says so. Fitted models are saved to `models/` and loaded on reruns.

In [13]:
import joblib

MODELS_DIR = Path("..") / "models"
MODELS_DIR.mkdir(exist_ok=True)
PARAM_GRID = {"tfidf__ngram_range": [(1, 1), (1, 2)], "tfidf__min_df": [2, 5], "ridge__alpha": [0.3, 1.0, 3.0, 10.0]}

text_models = {}
for name in VARIANTS_TXT:
    path = MODELS_DIR / f"tfidf_ridge_{name}_{VERSION}.joblib"
    if path.exists():
        print(f"{path} already exists, loading.")
        text_models[name] = joblib.load(path)
        continue
    start = time.time()
    search = tm.fit_tfidf_ridge(texts[name].loc[train_ids], Y.loc[train_ids], PARAM_GRID, cv=3, seed=42)
    text_models[name] = search.best_estimator_
    joblib.dump(search.best_estimator_, path)
    best = search.best_params_
    edge = best["ridge__alpha"] in (min(PARAM_GRID["ridge__alpha"]), max(PARAM_GRID["ridge__alpha"]))
    write_metadata(path, model="TfidfVectorizer(sublinear_tf) + Ridge", text=name, train_movies=len(train_ids),
                   targets=ATTRS, best_params={k: str(v) for k, v in best.items()},
                   cv_mean_spearman=round(search.best_score_, 4), param_grid={k: str(v) for k, v in PARAM_GRID.items()},
                   alpha_on_grid_edge=edge, seed=42, runtime_seconds=round(time.time() - start, 1))
    print(f"{name}: best {best}, cross-validated mean Spearman {search.best_score_:.3f}, {time.time() - start:.0f}s"
          + ("  <- alpha on the edge of the grid; widen it" if edge else ""))

../models/tfidf_ridge_plot_v1.joblib already exists, loading.
../models/tfidf_ridge_plot_reception_v1.joblib already exists, loading.
../models/tfidf_ridge_plot_reception_noscores_v1.joblib already exists, loading.


### Score on the held-out test movies

In [14]:
truth_test = Y.loc[test_ids]
genres_by_id = tiers.set_index("movieId")["genres"].combine_first(pd.read_csv(ML_DIR / "movies.csv").set_index("movieId")["genres"])
preds = {"genre_baseline": ve.genre_baseline(attr_scores[ATTRS], genres_by_id, list(test_ids))[ATTRS]}
for name in VARIANTS_TXT:
    preds[name] = tm.predict_frame(text_models[name], texts[name].loc[test_ids], test_ids, ATTRS)

# each model against the one below it; agreement() reports Spearman for both and a paired bootstrap interval
comparisons = {"plot vs genre_baseline": ("plot", "genre_baseline"),
               "plot_reception vs genre_baseline": ("plot_reception", "genre_baseline"),
               "plot_reception vs plot": ("plot_reception", "plot"),
               "plot_reception_noscores vs plot": ("plot_reception_noscores", "plot"),
               "plot_reception_noscores vs plot_reception": ("plot_reception_noscores", "plot_reception")}
EVAL_PATH = Path("..") / "artifacts" / "text_model_eval_v2.csv"          # v2: adds the no-review-scores model
rows = []
for label, (model, base) in comparisons.items():
    r = ve.agreement(preds[model], truth_test, baseline=preds[base], n_boot=1000, seed=42)
    r.insert(0, "comparison", label)
    rows.append(r)
    print(f"{label}: median Spearman {r['llm_rho'].median():.3f} vs {r['baseline_rho'].median():.3f}; "
          f"clearly better on {r['beats_baseline'].sum()} of {len(r)} vibes, clearly worse on {(r['diff_hi'] < 0).sum()}")
evaluation = pd.concat(rows).rename(columns={"llm_rho": "model_rho"})
evaluation.to_csv(EVAL_PATH, index=False)
write_metadata(EVAL_PATH, test_movies=len(test_ids), n_boot=1000, seed=42,
               note="Spearman per vibe on held-out test movies; diff_lo/hi = 95% paired bootstrap interval of model minus baseline")

plot vs genre_baseline: median Spearman 0.547 vs 0.370; clearly better on 36 of 41 vibes, clearly worse on 1
plot_reception vs genre_baseline: median Spearman 0.609 vs 0.370; clearly better on 41 of 41 vibes, clearly worse on 0
plot_reception vs plot: median Spearman 0.609 vs 0.547; clearly better on 41 of 41 vibes, clearly worse on 0
plot_reception_noscores vs plot: median Spearman 0.600 vs 0.547; clearly better on 40 of 41 vibes, clearly worse on 0
plot_reception_noscores vs plot_reception: median Spearman 0.600 vs 0.609; clearly better on 0 of 41 vibes, clearly worse on 33


### Quality-leak check: do the vibe scores secretly measure how good the movie is?

For each vibe, compare two correlations on the test movies:
- how strongly the **true** Genome score follows the movie's MovieLens average rating (some of that is real: acclaimed movies *are* more often slow or melancholic);
- how strongly each **model's** score follows it.

If a model follows rating much more than the truth does, it is using quality words as a shortcut. The column `excess` is model minus truth, so 0 means no extra quality signal.

In [15]:
from scipy.stats import spearmanr

rating = tiers.set_index("movieId").loc[test_ids, "mean_rating"]
leak = pd.DataFrame({"truth": [spearmanr(truth_test[a], rating)[0] for a in ATTRS]}, index=ATTRS)
for name in VARIANTS_TXT:
    leak[name] = [spearmanr(preds[name][a], rating)[0] for a in ATTRS]
    leak[f"excess_{name}"] = leak[name] - leak["truth"]
print("median absolute excess correlation with rating:",
      {n: round(leak[f"excess_{n}"].abs().median(), 3) for n in VARIANTS_TXT})
print(leak.loc[leak["excess_plot_reception"].abs().sort_values(ascending=False).index[:8],
               ["truth", "plot", "plot_reception", "plot_reception_noscores"]].round(2).to_string())
for a in ["dark_bleak", "slow", "melancholic"]:
    pos, neg = tm.top_words(text_models["plot_reception_noscores"], ATTRS.index(a), 8)
    print(f"no-scores model, {a:12s} + {', '.join(pos)}\n{'':28s} - {', '.join(neg)}")

median absolute excess correlation with rating: {'plot': np.float64(0.121), 'plot_reception': np.float64(0.05), 'plot_reception_noscores': np.float64(0.059)}
               truth  plot  plot_reception  plot_reception_noscores
christmas      -0.48 -0.07           -0.15                    -0.14
funny           0.05 -0.17           -0.13                    -0.13
midlife         0.07  0.11            0.21                     0.19
twist           0.19 -0.02            0.09                     0.07
atmospheric     0.54  0.22            0.44                     0.39
revenge        -0.06 -0.16           -0.16                    -0.15
non_hollywood   0.26  0.22            0.36                     0.33
satire         -0.12 -0.23           -0.22                    -0.22
no-scores model, dark_bleak   + violence, film festival, films, violent, suicide, list, apartment, drugs
                             - comedy, million, worst, love, in love, team, star, fun
no-scores model, slow         + cannes,

**Results (2,511 held-out test movies)**

| Model | Median Spearman over 41 vibes | Against the model below it |
|---|---|---|
| 0 Genre-only guess | 0.370 | — |
| 1 Plot | 0.547 | better than genre on 36 of 41 vibes, worse on 1 |
| **2 Plot + reception** | **0.609** | better than plot-only on **41 of 41** |
| 3 Plot + reception, no score sentences | 0.600 | worse than model 2 on 33 of 41 (small gaps) |

- **The text model clearly works.** It beats the genre-only guess on every vibe. It is strongest on dark_bleak (0.77), melancholic (0.77), slow (0.76) and atmospheric (0.74), and weakest on christmas (0.29), non-hollywood ending (0.39), space and future (0.41).
- **Critics' text adds real signal on every vibe** (0.547 → 0.609). This supports the idea that reception carries the "how it feels" part.
- **Quality leak: none beyond what the Genome already contains.** Some Genome vibes truly go with acclaimed films (atmospheric correlates 0.54 with rating in the true scores). The plot + reception model follows rating *less* than the truth does (0.44 for atmospheric), with a median excess of only 0.05. Removing the score sentences did not reduce this (0.059), because the link moves to other words such as "Cannes" and "festival". It also lowered accuracy slightly, so model 3 does not earn its place.
- **Chosen: model 2, plot + reception.** Settings picked by cross-validation: single words and two-word phrases, words in at least 2 texts, alpha = 1. alpha is inside its grid. min_df = 2 is on the low edge, but anything lower would mainly add one-off names and typos.

**Limitations**
- **The ceiling is the Genome.** The model learns the Genome's view, including its tie between some vibes and acclaim; the ranker (Step 10) should know vibe scores carry some quality signal.
- **Scope of the test.** Tested on movies that have Genome scores, which are better-known movies (all have 28+ ratings). Obscure movies have shorter Wikipedia texts and less reception, so expect lower accuracy there.
- **Rare vibes are weak** (christmas, space, future), probably too few examples.

### Vibe scores for every movie with text

- **Movies with Genome scores:** keep the real Genome scores. They are measured from many users' tags, so they are more reliable than any prediction.
- **Movies without Genome scores but with a Wikipedia plot or reception:** use model 2's predictions.
- **The `source` column** records which one each movie got, so later steps can weight them differently.
- **Movies with neither (no Wikipedia text):** no score yet. They would need TMDB's short overview, a later step.

Predictions are kept on the same 0–1 scale, clipped at the edges, because ridge can go slightly outside it.

In [16]:
VIBE_PATH = PROCESSED_DIR / f"vibe_scores_{VERSION}.parquet"
if VIBE_PATH.exists():
    print(f"{VIBE_PATH} already exists, loading.")
    vibe_scores = pd.read_parquet(VIBE_PATH)
else:
    pool_ids = pool["movieId"]                                               # tier 2 + 3 movies
    has_text = wt.index[(wt["plot_words"] > 0) | (wt["reception_words"] > 0)]
    from_genome = pool_ids[pool_ids.isin(attr_scores.index)]
    from_model = pool_ids[~pool_ids.isin(attr_scores.index) & pool_ids.isin(has_text)]
    model_text = pd.Series({i: tm.movie_text(wt.at[i, "plot"], wt.at[i, "reception"]) for i in from_model})
    predicted = tm.predict_frame(text_models["plot_reception"], model_text, model_text.index, ATTRS).clip(0, 1)
    vibe_scores = pd.concat([attr_scores.loc[from_genome, ATTRS].assign(source="genome"),
                             predicted.assign(source="text_model")])
    vibe_scores.index.name = "movieId"
    vibe_scores = vibe_scores.reset_index()
    vibe_scores.to_parquet(VIBE_PATH, index=False)
    write_metadata(VIBE_PATH, rows=len(vibe_scores), attributes=ATTRS,
                   sources=vibe_scores["source"].value_counts().to_dict(),
                   model=f"models/tfidf_ridge_plot_reception_{VERSION}.joblib (test median Spearman 0.609 vs genre 0.370)",
                   note="Genome scores where available, else text-model predictions clipped to 0-1; tier 2 + 3 movies only")
    print(f"saved {len(vibe_scores):,} rows to {VIBE_PATH}")

tier_of = tiers.set_index("movieId")["tier"]
summary = vibe_scores.assign(tier=vibe_scores["movieId"].map(tier_of)).groupby(["tier", "source"]).size().unstack(fill_value=0)
summary["no_vibe_scores"] = pool.groupby("tier").size() - summary.sum(axis=1)
print(summary)

../data/processed/vibe_scores_v1.parquet already exists, loading.
source  genome  text_model  no_vibe_scores
tier                                      
2            0        2904            2287
3        12877       16274            5252
